# TP2 与 FSDP2 的 TPS/显存对比实验

本节用于补充 07.05 的 FSDP 优势分析。两条路线使用同一份 Wordle SFT 数据、Qwen3-1.7B、bf16、seq_len、global batch、packing、activation checkpoint、compile 选项和连续稳态 step；唯一变化是并行策略：TP degree=2 或 FSDP2 `dp_shard=2`。所有测试固定 **2 张 NPU**。

## 实验协议与证据

两条路线使用同一个训练入口 `sft_qwen3_1_7b_wordle`（Qwen3-1.7B、`assets/data/wordle`）和同一组训练参数：bf16、`seq_len=1024`、`global_batch_size=4`、相同 seed、12 个连续 step；唯一变化是并行策略——FSDP2（`dp_shard=2`）或 TP2（`tensor_parallel=2`）。所有测试固定 **2 张 NPU**。

采集设置：`--metrics.log-freq 1` 让两个 rank 的日志逐 step 记录 `loss` / `grad_norm` / `memory` / `tps`；同一个 profiler 周期（`warmup=3, active=1, repeat=1, skip_first=1`）在每个 rank 上抓取 1 个 step 的算子级 trace。

需要提前说明一处由并行策略自身带来的差异：FSDP2 路线的数据并行度是 2、梯度累积为 1 个 micro-batch；TP2 路线的数据并行度是 1、同样的 `global_batch_size=4` 会落成 2 个梯度累积 micro-batch。`tps` 是 per-device 口径（已除以 `non_data_parallel_size`），两条路线仍然可以直接比较，但解读吞吐时要知道这一点。

读数口径：

- **稳态吞吐**：丢弃前 2 个编译/warmup step，在 steps 3–12 上取 `tps` 的中位数。`tps` 是 torchtitan 的 per-device 吞吐（`labels.numel() / (time_delta × non_data_parallel_size)`），已排除模型并行维，两条路线可直接比较；profiler 周期会让窗口内 2 个 step 偏低，median 对这两个离群点不敏感。
- **显存**：训练日志 `memory:` 字段的双 rank 峰值（`max_reserved`），同时报告 rank 之间的差。
- **通信**：每个 rank 的 `trace_view.json` 里集合通信（`Hccl*`）的调用次数与设备任务队列时长（`Dequeue@Hccl*`），并用同一 rank、同一 step 的墙钟时间换算占比。

本轮**不报告** `Stage` / `Communication(Not Overlapped)`：本环境 CANN 25.5.5 的在线解析只产出算子级 `trace_view.json`，不生成 `step_trace_time.csv`（与 08.03 §5 的口径一致）。因此通信证据是"调用次数 + 设备队列时长 + 占 step 比例"，而不是"未重叠通信时间"。

In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/gitCode/cann/torchtitan-npu-wordle-latest
os.environ.update(dict(line.strip().split('=',1) for line in os.popen('source /home/developer/Ascend/cann/set_env.sh && env') if '=' in line));
os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/local/bin:/usr/local/sbin"


In [ ]:
!pip list | grep torch

In [ ]:
%%bash
set -euo pipefail
export PATH="/home/developer/.virtualenvs/python312/bin:$PATH"
cd /mnt/workspace/gitCode/cann/torchtitan-npu-wordle-latest

# 两条路线各跑一次，输出目录互不覆盖；训练日志留在各自的采集目录里作为证据。
rm -rf outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing outputs/profile_traces/tp_fsdp_tp2_2npu_timing
mkdir -p outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing outputs/profile_traces/tp_fsdp_tp2_2npu_timing

# 两条路线完全相同的参数：数据、dtype、seq_len、GBS、日志频率、profiler 周期。
common=(
  --module torchtitan_npu.models.qwen3.config_registry
  --config sft_qwen3_1_7b_wordle
  --training.steps 12 --training.seq-len 1024 --training.dtype bfloat16
  --training.global-batch-size 4
  --metrics.log-freq 1
  --parallelism.data-parallel-replicate-degree 1 --parallelism.context-parallel-degree 1
  --profiler.enable-profiling --profiler.profile-freq 4 --profiler.profiler-warmup 3
  --profiler.profiler-active 1 --profiler.profiler-repeat 1 --profiler.profiler-skip-first 1
  --dataloader.dataset-path assets/data/wordle
  --override.imports
  'torchtitan_npu.override.common.profiler.cann={"profile_ranks":[-1],"profile_with_memory":false}'
)

echo '=== 路线 A：FSDP2（dp_shard=2, tp=1） ==='
# 与同机其它实验隔离端口：HCCL_IF_BASE_PORT 与 NPU socket 端口段都各用一份。
HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 HCCL_IF_BASE_PORT=33500 \
  HCCL_NPU_SOCKET_PORT_RANGE=62000-62015 LOG_RANK=0,1 NGPU=2 \
  bash scripts/run_train.sh "${common[@]}" \
    --parallelism.data-parallel-shard-degree 2 \
    --profiler.save-traces-folder profile_traces/tp_fsdp_fsdp2_2npu_timing \
  > outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing/train.log 2>&1
grep -E 'step: +12 ' outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing/train.log \
  | sed 's/\x1b\[[0-9;]*m//g' | tail -2

echo '=== 路线 B：TP2（tp=2, dp_shard=1） ==='
HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 HCCL_IF_BASE_PORT=33510 \
  HCCL_NPU_SOCKET_PORT_RANGE=62016-62031 LOG_RANK=0,1 NGPU=2 \
  bash scripts/run_train.sh "${common[@]}" \
    --parallelism.data-parallel-shard-degree 1 --parallelism.tensor-parallel-degree 2 \
    --profiler.save-traces-folder profile_traces/tp_fsdp_tp2_2npu_timing \
  > outputs/profile_traces/tp_fsdp_tp2_2npu_timing/train.log 2>&1
grep -E 'step: +12 ' outputs/profile_traces/tp_fsdp_tp2_2npu_timing/train.log \
  | sed 's/\x1b\[[0-9;]*m//g' | tail -2

echo '完成：train.log 与双 rank trace 分别位于 outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing/ 与 outputs/profile_traces/tp_fsdp_tp2_2npu_timing/。'
echo '注意：--override.imports 一旦在命令行给出就会替换 recipe 的整个列表；本 recipe（flex 路线）没有内置 override，因此这里只列 profiler 一条。'

## 理论分析与可检验读数

### 理论预期


FSDP2 将参数、梯度和优化器状态分片。前向的逐层 all-gather 与反向的逐层 reduce-scatter 可以在相邻层计算存在时被流水隐藏；关键风险是参数 unshard/reshard 的布局与搬运开销。TP2 则把 Attention/MLP 内的张量维度切分，降低单卡权重与部分激活占用，但在层内依赖边界引入 all-reduce、all-gather 或 redistribute。这些同步通常直接阻塞下一段计算，因此通信暴露会随层数累积。

### 如何在算子级 trace 中验证

当前 timing trace 使用 `profile_with_stack=false`，`with_stack=true` 会增加调用栈采集开销并扰动 profiling 时间，不能与本轮真实时间混用。

本节只建立理论预期，不重复采集证据。可检验的读数放在后面：双 rank 训练日志的稳态 `tps` 与峰值显存，以及每个 rank 的 `trace_view.json` 中 collective 的调用次数、设备任务队列时长（`Dequeue@Hccl*`）与占 step 墙钟的比例。本轮 trace 是算子级的：`Communication(Not Overlapped)` 这类关键路径读数在本环境不可得，所以理论中的“重叠窗口”只能给出结构判断，不能靠本 notebook 的读数用数值证实。

### 证据解释边界

- **真实时间**以训练日志为准：`tps` 是 per-device 稳态吞吐，`memory` 是日志窗口内的峰值 reserved；两者都取双 rank，避免单 rank 掩盖差异。
- **通信**只能报"调用次数 + 设备任务队列时长 + 占 step 墙钟比例"。`Dequeue@Hccl*` 是设备流上各 collective 的执行时长之和，**不含跨 rank 等待，也没有与计算求交集**，因此它是暴露上界，不是"未重叠通信"。
- 本轮 trace 是算子级（`profiler_level=Level1`、`profile_with_stack=false`、`profile_with_memory=false`），只产出 `trace_view.json`：没有 `step_trace_time.csv`，没有 `communication.json`，也没有设备侧 kernel 时间线。因此**不做** Attention/MLP/Norm 的模块级归因，也不把 collective 时长相加当成 step time。
- 只有双 rank 的日志与 trace 都存在时才写结论；trace 中的 `ProfilerStep#4` 表明本轮 profiling 落在第 4 个 step。

## 已完成的 2NPU 实测

两条路线在同一台机器上顺序执行（避免互相占卡），共用同一份 Wordle parquet、Qwen3-1.7B、bf16、`seq_len=1024`、`global_batch_size=4`、12 steps，并用 `LOG_RANK=0,1` 让两个 rank 都逐 step 写日志。下表取自 `train.log`（steps 3–12 的 median，前 2 个编译 step 已丢弃）与双 rank 的 `trace_view.json`。

| 指标 | FSDP2（`dp_shard=2`） | TP2（`tensor_parallel=2`） |
|---|---|---|
| 稳态 `tps` 中位数（rank0 / rank1） | 1615 / 1614.5 | 560.5 / 560.5 |
| 峰值 `memory`（rank0 / rank1） | 12.57 / 12.57 GiB | 10.67 / 10.65 GiB |
| 12 步 loss | 12.369 → 11.549 | 12.282 → 11.518 |
| profiled step 的 collective 次数（rank0） | 59 AllGather + 30 ReduceScatter + 5 AllReduce = 94 | 340 AllGather + 228 ReduceScatter + 275 AllReduce = 843 |
| 设备队列时长合计（rank0 / rank1） | 18.51 / 16.42 ms | 133.39 / 128.41 ms |
| step 墙钟（由该 rank 的 median `tps` 反推） | 1268 / 1269 ms | 3654 / 3654 ms |
| 集合通信设备时长占 step | 1.46% / 1.29% | 3.65% / 3.51% |

两条路线的 rank 间差异都很小（`tps` 差 <0.2%，显存差 ≤0.04 GiB），说明观测是稳定的（重复运行时 `tps` 会有几个百分点的波动，因此结论按量级表述）。在这个固定配置下：

- **吞吐**：FSDP2 的稳态 `tps` 中位数（1614.8）是 TP2（560.5）的 **2.88×**；反推的 step 墙钟为 1.27 s vs 3.65 s，而两条路线每个 step 处理的是同一份 4×1024 个 slot token。
- **显存**：TP2 的峰值低约 **15%**（10.67 GiB vs 12.57 GiB），远小于“权重与激活都切半”的 2 倍直觉；常驻显存不是本 workload 的瓶颈（单卡 61.27 GiB）。
- **通信**：TP2 的 collective 调用次数（843）是 FSDP2（94）的 **9.0 倍**，设备队列时长（128–133 ms）是 FSDP2（16.42–18.51 ms）的 **7.2–7.8 倍**；方向与理论预期一致。
- 两条路线都跑完 12 步且 loss 整体下降，说明本 workload 上两条路线都可用，差别在性能与显存。

补充两点口径：
1. TP2 的 `HcclAllReduce` 次数（275）远高于 FSDP2（5），对应 TP 层内对激活/梯度的 AllReduce 与 DTensor 重分布；FSDP2 的通信以逐层 AllGather + ReduceScatter 为主（59 + 30）。
2. 两次运行的 profiler 周期都落在第 4 个 step（trace 里只有 `ProfilerStep#4`），被 profiler 影响的 2 个 step 的 `tps` 偏低，median 不受影响。本轮**没有**出现“TP/PP 混合精度被禁用”的日志：两条路线都是 `training.dtype=bfloat16` 的全 bf16 配置，可以直接比较。

In [ ]:
# 2NPU 对比读数：只用本轮真实产物——双 rank 的 train.log 与每 rank 的算子级 trace_view.json。
import json
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path('outputs/profile_traces')          # profiler 的 base_folder 是 job.dump_folder=outputs
RUNS = {
    'fsdp2': ROOT / 'tp_fsdp_fsdp2_2npu_timing',
    'tp2': ROOT / 'tp_fsdp_tp2_2npu_timing',
}
STEADY = range(3, 13)                          # 丢弃前 2 个编译 step；profiler 周期的 2 个离群 step 由 median 吸收
GBS, SEQ_LEN = 4, 1024
# 由 tps 反推 step 墙钟：tps 是 per-device 吞吐，rank 每 step 处理 GBS×seq_len/dp_degree 个 token，
# 再按 non_data_parallel_size（tp/cp/pp）归一化，因此 step = GBS×seq_len / (dp_degree × ndp × tps)。
ROUTE_META = {'fsdp2': {'dp_degree': 2, 'ndp': 1}, 'tp2': {'dp_degree': 1, 'ndp': 2}}

ANSI = re.compile(r'\x1b\[[0-9;]*m')
STEP_RE = re.compile(
    r'\[rank(?P<rank>\d+)\].*?step:\s*(?P<step>\d+)\s+loss:\s*(?P<loss>[\d.]+)\s+'
    r'grad_norm:\s*(?P<gn>[\d.]+)\s+memory:\s*(?P<mem>[\d.]+)GiB.*?tps:\s*(?P<tps>[\d,]+)'
)
PID_RE = re.compile(
    r'\[rank(?P<rank>\d+)\]:I\d+ [\d:.]+ (?P<pid>\d+) torch_npu/utils/patch_getenv\.py:15\] '
    r'get env LOCAL_RANK = (?P<local>\d+)'
)

def load_trace(path):
    """CANN 的 trace_view.json 可能带尾逗号，也可能在采集结束时被截断最后一个对象。"""
    raw = path.read_text(errors='replace')
    raw = re.sub(r',(\s*[}\]])', r'\1', raw)
    if not raw.rstrip().endswith(']'):
        raw = raw.rstrip().rstrip(',') + ']'
    return json.loads(raw)

def read_steps(log_path):
    """返回 {rank: {step: {tps, mem, loss}}} 与日志头部解析出的 pid→rank 映射。"""
    text = ANSI.sub('', log_path.read_text(errors='replace'))
    pid2rank = {int(m['pid']): int(m['rank']) for m in PID_RE.finditer(text)}
    steps = defaultdict(dict)
    for m in STEP_RE.finditer(text):
        steps[int(m['rank'])][int(m['step'])] = {
            'tps': float(m['tps'].replace(',', '')),
            'mem': float(m['mem']),
            'loss': float(m['loss']),
        }
    return steps, pid2rank

def collective_stats(trace_path):
    """统计一个 rank 的 profiled step 中集合通信的次数与设备任务队列时长。"""
    counts, device_us, step_names = Counter(), defaultdict(float), set()
    for event in load_trace(trace_path):
        name = event.get('name', '')
        if name.startswith('ProfilerStep'):
            step_names.add(name)
        if name.startswith('Dequeue@Hccl'):
            op = name.split('@', 1)[1]
            counts[op] += 1
            device_us[op] += float(event.get('dur', 0))
    return counts, device_us, sorted(step_names)

def trace_rank(trace_path, pid2rank):
    """trace 目录名形如 <host>_<pid>_<timestamp>_ascend_pt，用日志里的 pid→rank 映射认 rank。"""
    pid = int(trace_path.parent.parent.name.split('_')[1])
    return pid2rank[pid]

report = {}
for route, root in RUNS.items():
    steps, pid2rank = read_steps(root / 'train.log')
    assert sorted(steps) == [0, 1], f'{route}: 需要双 rank 逐 step 日志，实际只有 {sorted(steps)}'
    report[route] = {'steps': steps, 'trace': {}}
    print(f'=== {route}（pid→rank: {pid2rank}）===')
    for rank in sorted(steps):
        rows = steps[rank]
        steady = [rows[s]['tps'] for s in STEADY if s in rows]
        median_tps = statistics.median(steady)
        peak_mem = max(row['mem'] for row in rows.values())
        meta = ROUTE_META[route]
        step_ms = GBS * SEQ_LEN / (meta['dp_degree'] * meta['ndp'] * median_tps) * 1000
        report[route][rank] = {'median_tps': median_tps, 'peak_mem': peak_mem, 'step_ms': step_ms}
        print(f'  rank{rank}: steps={len(rows)}, median_tps={median_tps:.1f}, '
              f'peak_mem={peak_mem:.2f} GiB, step={step_ms:.0f} ms, '
              f'loss {rows[min(rows)]["loss"]:.3f} -> {rows[max(rows)]["loss"]:.3f}')
        print(f'          steady_tps={steady}')
    for trace_path in sorted(root.rglob('trace_view.json')):
        rank = trace_rank(trace_path, pid2rank)
        counts, device_us, profiled = collective_stats(trace_path)
        total_ms = sum(device_us.values()) / 1000
        step_ms = report[route][rank]['step_ms']
        report[route]['trace'][rank] = {'counts': dict(counts), 'device_ms': total_ms, 'profiled': profiled}
        print(f'  rank{rank} trace {profiled}: 合计 {sum(counts.values())} 次 {dict(counts)}')
        print(f'          设备队列时长 {total_ms:.2f} ms '
              f'（{ {k: round(v / 1000, 2) for k, v in sorted(device_us.items())} } ms）'
              f' 占 step {step_ms:.0f} ms 的 {total_ms / step_ms:.2%}')

f_tps = statistics.median([report['fsdp2'][r]['median_tps'] for r in (0, 1)])
t_tps = statistics.median([report['tp2'][r]['median_tps'] for r in (0, 1)])
f_mem = max(report['fsdp2'][r]['peak_mem'] for r in (0, 1))
t_mem = max(report['tp2'][r]['peak_mem'] for r in (0, 1))
f_comm = [report['fsdp2']['trace'][r]['device_ms'] for r in (0, 1)]
t_comm = [report['tp2']['trace'][r]['device_ms'] for r in (0, 1)]
f_ops = [sum(report['fsdp2']['trace'][r]['counts'].values()) for r in (0, 1)]
t_ops = [sum(report['tp2']['trace'][r]['counts'].values()) for r in (0, 1)]
print()
print(f'稳态 tps 中位数：FSDP2 {f_tps:.1f} vs TP2 {t_tps:.1f} -> {f_tps / t_tps:.2f}x')
print(f'峰值显存：FSDP2 {f_mem:.2f} GiB vs TP2 {t_mem:.2f} GiB -> TP2 低 {(1 - t_mem / f_mem):.1%}')
print(f'集合通信次数（rank0/rank1）：FSDP2 {f_ops[0]}/{f_ops[1]}，TP2 {t_ops[0]}/{t_ops[1]}'
      f' -> {t_ops[0] / f_ops[0]:.1f}x / {t_ops[1] / f_ops[1]:.1f}x')
print(f'集合通信设备队列时长（rank0/rank1）：FSDP2 {f_comm[0]:.2f}/{f_comm[1]:.2f} ms，'
      f'TP2 {t_comm[0]:.2f}/{t_comm[1]:.2f} ms -> {t_comm[0] / f_comm[0]:.1f}x / {t_comm[1] / f_comm[1]:.1f}x')


## 理论与实测的对应结论

| 理论命题 | 本轮实测信号 | 本轮结论 |
|---|---|---|
| FSDP2 的逐层参数通信可被计算隐藏 | 每 step 94 次 collective，设备队列时长合计 16.42–18.51 ms，占 step 墙钟 1.29%–1.46% | 通信量级很小；是否与计算重叠本轮无法直接判定（缺设备时间线） |
| TP2 的层内同步更容易落在关键路径上 | 每 step 843 次 collective，设备队列时长合计 128.41–133.39 ms，占 step 墙钟 3.51%–3.65% | 调用次数与设备时长都比 FSDP2 高约 7–9 倍，但合计只能解释 2.39 s 墙钟差中的约 0.12 s，剩余部分需要设备时间线定位 |
| TP2 以显存换吞吐 | 峰值显存 10.67 GiB（低约 15%），稳态 `tps` 560.5（约 1/2.88） | 本配置下换来的显存收益有限，代价是约 2.9 倍吞吐 |

这张表只给出“理论 → 证据”的阅读路径。`Stage` 与 `Communication(Not Overlapped)` 需要 `step_trace_time.csv`，本轮产物里没有；collective 的设备队列时长含排队语义，不能相加当作 step time。

## FSDP2 通信与计算重叠：图与口径

下面的两张图来自课程材料中 `with_stack=false` 的 FSDP2 timing trace，用于说明“逐层参数交换与相邻计算在时间线上相邻、存在重叠窗口”这一结构事实。本轮采集的同类文件在 `outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing/<host>_<pid>_<ts>_ascend_pt/ASCEND_PROFILER_OUTPUT/trace_view.json`，可以用 ui.perfetto.dev 打开查看逐层算子时间线。

![FSDP2 通信与计算重叠分析](./images/08.04_overlap_analysis.png)

图：FSDP2 时间线中 all-gather/reduce-scatter 与相邻层计算的相邻关系。

![Matmul 与 AllGather 的时序对比](./images/08.04_matmul_vs_allgather.png)

图：同一时间窗内 Matmul 与 HcclAllGather 的时序对比：all-gather 本身短于相邻 Matmul，具备被计算隐藏的结构条件。

## 结论：本课程选择 FSDP2

在本课程固定的 2NPU Wordle SFT workload 下，选择 FSDP2，不选择 TP2。

依据本轮实测：FSDP2 的稳态 per-device `tps` 中位数是 1614.8，TP2 是 560.5（**2.88×**）；反推 step 墙钟 1.27 s vs 3.65 s。TP2 的常驻显存更低（10.67 GiB vs 12.57 GiB，约 15%），但在单卡 61.27 GiB 的本 workload 里不构成收益。两个 rank 的日志与 trace 都完整，rank 间差异 <0.2%。

通信账本的方向与理论预期一致：TP2 每 step 的 collective 调用次数（843）是 FSDP2（94）的 9.0 倍，设备队列时长（128.41–133.39 ms）是 FSDP2（16.42–18.51 ms）的 7.2–7.8 倍。但边界必须说清：**本轮没有设备时间线，无法给出“未重叠通信”时间**；设备队列时长合计只占 TP2 step 墙钟的 3.51%–3.65%、FSDP2 的 1.29%–1.46%，不足以单独解释 2.9 倍的吞吐差。因此本节结论是“本配置下 FSDP2 吞吐更高、TP2 显存更低、TP2 的集合通信高一个量级”；至于“TP2 慢主要因为通信暴露”，仍是一个需要设备时间线验证的假设，而不是本轮证据的结论。

这个结论只用于本课程的工程选型，不把 FSDP2 宣称为所有模型、序列长度和硬件规模下都优于 TP。

## 本章小结

本章用同一条证据链回答工程选型：固定 Qwen3-1.7B Wordle SFT、bf16、2 张 Ascend NPU。**CP 是容量工具，不是默认的吞吐加速**；但“CP 能不能用”首先是版本问题：当前依赖版本里 Qwen3 TND + CP 会被上游直接拒绝，而本 workload 的 no-CP 也已不再 OOM（`peak reserved` 32.58 GiB ≈ 单卡 53%）。因此第 8 章把结论限定为“能力边界 + 容量基线 + 通信量级外推”（08.02/08.03）。并行策略对比（08.04）显示 FSDP2 稳态吞吐约为 TP2 的 2.88 倍，TP2 常驻显存约低 15%，且 TP2 每 step 的集合通信次数与设备队列时长都高约 7–9 倍；本轮 trace 为算子级，因此不报告 `Communication(Not Overlapped)`。

## 练习

1. （判断题）本实验固定 2 张 NPU、同一份数据与同一组训练参数，只切换 FSDP2（`dp_shard=2`）与 TP2（`tensor_parallel=2`）；实测 FSDP2 的稳态 `tps` 更高，而 TP2 的日志峰值显存更低。

2. （多选题）本轮证据支持下列哪些说法？
    A. FSDP2 的稳态 `tps` 中位数约为 TP2 的 2.88 倍
    B. TP2 每 step 的 collective 调用次数约为 FSDP2 的 9 倍，设备队列时长约为 7–8 倍
    C. 集合通信的设备队列时长合计占 step 墙钟比例很小，因此本轮的吞吐差不能只归因于通信
    D. 本轮 trace 给出了 `Communication(Not Overlapped)`，可以据此断言 TP2 的通信全部暴露在关键路径上

3. （简答题）为什么本 notebook 不报告 `Stage` 与 `Communication(Not Overlapped)`？要得到这两个量需要补什么采集？

In [ ]:
%cd $original_dir


In [ ]:
!cat ./answer/08.04_answer.txt
